In [ ]:
!pip install open_clip_torch

import open_clip
import torch
from PIL import Image
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai"
)

model.eval()

image = preprocess(Image.open(image_path)).unsqueeze(0)

with torch.no_grad():
    features = model.encode_image(image)

embedding = features.cpu().numpy().flatten().tolist()

print("CLIP Embedding Length:", len(embedding))
print("CLIP Output:", embedding)

In [ ]:
!pip install transformers

from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from google.colab import files

# Upload Image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Load BLIP
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
# Load image
image = Image.open(image_path).convert("RGB")
inputs = processor(image, return_tensors="pt")
output = model.generate(**inputs)
caption = processor.decode(output[0], skip_special_tokens=True)

print("BLIP Caption:", caption)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
model = YOLO("yolov8n.pt")
results = model.predict(
    source=image_path,
    conf=0.05,
    imgsz=640,
    save=False
)
img = cv2.imread(image_path)# Read image
detections = []
for r in results:
    if r.boxes is None:
        print("No detections found")
        continue
    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()
    for box, conf, cls in zip(boxes, confs, classes):
        label = model.names[int(cls)]
        x1, y1, x2, y2 = map(int, box)
        detections.append({
            "label": label,
            "confidence": float(conf),
            "bbox": [x1, y1, x2, y2]
        })
        cv2.rectangle(img, (x1,y1),(x2,y2),(0,255,0),2)  # draw bounding box
        cv2.putText(img,label,(x1,y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,(0,255,0),2)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
print("Detections:", detections)